# Zero-shot ROCOv2 captioning --- BioMedVQA (BioMedCLIP + GPT-2)

The notebook counterpart of `caption_roco_biomed.py`. It reuses the **BioMedVQA
model fine-tuned on VQA-RAD** (frozen BioMedCLIP vision encoder $\to$ a linear
translator $\to$ GPT-2) for **zero-shot** captioning on ROCOv2 --- the direct
parallel to the BLIP-2 zero-shot run.

**Same conditions as BLIP-2, for comparability:** the `"a photo of"` prompt and
the identical beam-search decoding (`num_beams=5`, `no_repeat_ngram_size=3`,
`min/max_new_tokens=8/40`), scored with the same five-metric suite.

**Expect a weak baseline.** This model hands the LLM only **one** visual token
(the 512-d BioMedCLIP CLS vector, projected to GPT-2's 768-d space), and the
decoder is a **general-domain GPT-2** with no medical pre-training --- so it
produces medical-sounding but ungrounded captions. That is the informative floor:
it isolates the cost of the one-token bottleneck + general LLM versus BLIP-2's 32
query tokens + OPT.

> The full 9,927-image run should be done headless via `caption_roco_biomed.py`
> in `tmux`; this notebook defaults to a 500-image subset for a quick number.

In [ ]:
import torch
import torch.nn as nn
from PIL import Image

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device =", device)

In [ ]:
# ── BioMedCLIP vision encoder + DETERMINISTIC eval transform ─────────────────
import open_clip
MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
biomedclip_model, _preprocess_train, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME)
vision_encoder = biomedclip_model.visual

# preprocess_val = Resize + CenterCrop (deterministic). The training notebook used
# preprocess_train (RandomResizedCrop), which is random -> non-reproducible captions
# and a partial crop. For a clean captioning metric we use the deterministic transform.
image_processor = preprocess_val
print("BioMedCLIP vision encoder + deterministic eval transform loaded")

## The model: captioning = the image as one visual token

`BioMedVQA` turns an image into a single ``word'': the frozen BioMedCLIP encoder
gives a 512-d vector, the trainable `translator` maps it into GPT-2's 768-d
embedding space, and it is prepended to the prompt embeddings. Captioning is the
same forward as VQA, only the prompt changes --- here `"a photo of"` instead of
`"Question: ... Answer:"` --- and GPT-2 completes it. The one-token visual
bottleneck is the model's main limitation.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

class BioMedVQA(nn.Module):
    def __init__(self, vision_encoder, text_model_name="gpt2"):
        super().__init__()
        self.vision_encoder = vision_encoder
        for p in self.vision_encoder.parameters():
            p.requires_grad = False
        self.llm = GPT2LMHeadModel.from_pretrained(text_model_name)      # base weights, overwritten on load
        self.tokenizer = GPT2Tokenizer.from_pretrained(text_model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.translator = nn.Linear(512, self.llm.config.hidden_size)    # 512 -> 768

    @torch.no_grad()
    def generate(self, images, input_ids, attention_mask, **gen_kwargs):
        image_features   = self.vision_encoder(images)                          # [B, 512]
        translated_image = self.translator(image_features).unsqueeze(1)         # [B, 1, 768]  (one visual token)
        text_embeddings  = self.llm.transformer.wte(input_ids)                  # [B, L, 768]
        combined = torch.cat([translated_image, text_embeddings], dim=1)        # [B, 1+L, 768]
        img_att  = torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
        combined_att = torch.cat([img_att, attention_mask], dim=1)
        return self.llm.generate(inputs_embeds=combined, attention_mask=combined_att, **gen_kwargs)

In [ ]:
# ── Build the model and load the VQA-RAD fine-tuned weights ──────────────────
model = BioMedVQA(vision_encoder).to(device)
ckpt = torch.load("/home/matei/biomed_vqa_checkpoints/biomed_vqa_final.pt", map_location=device)
model.translator.load_state_dict(ckpt["translator"])
model.llm.load_state_dict(ckpt["llm"])
model.eval()
print("loaded fine-tuned BioMedVQA weights (translator + GPT-2)")

In [ ]:
# ── ROCOv2 test split (image -> reference caption) ──────────────────────────
import os
import pandas as pd

ROCO_DIR = "/home/matei/rocov2"
IMG_DIR  = os.path.join(ROCO_DIR, "test")
caps = pd.read_csv(os.path.join(ROCO_DIR, "test_captions.csv")).dropna(subset=["Caption"]).reset_index(drop=True)
records = [{"id": r.ID, "path": os.path.join(IMG_DIR, f"{r.ID}.jpg"), "caption": str(r.Caption)}
           for r in caps.itertuples() if os.path.isfile(os.path.join(IMG_DIR, f"{r.ID}.jpg"))]
print(f"ROCOv2 test: {len(records)} image-caption pairs")
print("example:", records[0]["id"], "->", records[0]["caption"][:80])

In [ ]:
# ── Captioning function ("a photo of", identical decoding to the BLIP-2 run) ──
from tqdm.auto import tqdm

CAPTION_PROMPT = "a photo of"
CAPTION_BATCH  = 16
# IDENTICAL to caption_roco.py so the BioMedVQA and BLIP-2 numbers are comparable.
GEN_KWARGS = dict(max_new_tokens=40, min_new_tokens=8, num_beams=5,
                  no_repeat_ngram_size=3, length_penalty=1.0,
                  eos_token_id=model.tokenizer.eos_token_id,
                  pad_token_id=model.tokenizer.eos_token_id)

def _load_pixels(paths):
    imgs = [image_processor(Image.open(p).convert("RGB")) for p in paths]
    return torch.stack(imgs).to(device)

def caption_records(recs, prompt=CAPTION_PROMPT, batch_size=CAPTION_BATCH):
    tok = model.tokenizer(prompt, return_tensors="pt").to(device)   # same prompt for all -> no padding
    preds = []
    for i in tqdm(range(0, len(recs), batch_size), desc="captioning", leave=False):
        chunk = recs[i:i+batch_size]
        pix = _load_pixels([r["path"] for r in chunk])
        B = pix.shape[0]
        ids = tok.input_ids.expand(B, -1)
        att = tok.attention_mask.expand(B, -1)
        gen = model.generate(pix, ids, att, **GEN_KWARGS)
        preds.extend(s.strip() for s in model.tokenizer.batch_decode(gen, skip_special_tokens=True))
    return preds

In [ ]:
# ── Eyeball: what does the model caption? ──
sample = records[:8]
sp = caption_records(sample, batch_size=8)
for r, p in zip(sample, sp):
    print(f"REF : {r['caption'][:95]}")
    print(f"PRED: {p[:95]}\n")

In [ ]:
# ── Caption metrics: BLEU-1..4, METEOR, ROUGE-L, CIDEr, BERTScore ────────────
# BERTScore uses microsoft/deberta-xlarge-mnli (F1) -- the SAME model the
# ImageCLEF/ROCOv2 leaderboard uses -- so this BERTScore is directly comparable to
# the ROCOv2 paper baselines (roberta-large would sit on a different, higher scale).
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from nltk.translate.meteor_score import meteor_score

# deberta needs three fixes on this stack: (1) unblock its .bin load (the load is
# weights_only=True and safe); (2) cap the sentinel tokenizer max_length; (3) small
# batch (disentangled attention is memory-heavy). Built once here, reused per call.
import transformers.modeling_utils as _mu
_mu.check_torch_load_is_safe = lambda *a, **k: None
from bert_score import BERTScorer
BERT_MODEL = "microsoft/deberta-xlarge-mnli"
_bert_scorer = BERTScorer(model_type=BERT_MODEL, batch_size=8)
_bert_scorer._tokenizer.model_max_length = 512

def _norm(s):
    return " ".join(str(s).lower().split())

def compute_caption_metrics(preds, refs):
    # an empty caption crashes BERTScore's empty-string path -> replace with "."
    preds = [p if str(p).strip() else "." for p in preds]
    refs  = [r if str(r).strip() else "." for r in refs]
    gts = {i: [_norm(refs[i])]  for i in range(len(refs))}
    res = {i: [_norm(preds[i])] for i in range(len(preds))}
    bleu, _  = Bleu(4).compute_score(gts, res)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    meteor = sum(meteor_score([_norm(refs[i]).split()], _norm(preds[i]).split())
                 for i in range(len(preds))) / len(preds)
    _, _, F = _bert_scorer.score(preds, refs, batch_size=8, verbose=False)
    m = {"BLEU-1": bleu[0], "BLEU-2": bleu[1], "BLEU-3": bleu[2], "BLEU-4": bleu[3],
         "METEOR": meteor, "ROUGE-L": rouge, "CIDEr": cider,
         "BERTScore-F1": F.mean().item(), "BERTScore-model": BERT_MODEL}
    print("=" * 48)
    for k, v in m.items():
        print(f"  {k:16s}: {v:.4f}" if isinstance(v, float) else f"  {k:16s}: {v}")
    print("=" * 48)
    return m

In [ ]:
# ── Zero-shot captioning evaluation on ROCOv2 test ──
# Subset for a quick number; set SUBSET_N = None for the full 9,927 test images
# (better run headless via caption_roco_biomed.py in tmux).
SUBSET_N = 500

eval_recs = records if SUBSET_N is None else records[:SUBSET_N]
print(f"captioning {len(eval_recs)} images (prompt={CAPTION_PROMPT!r}) ...")
preds = caption_records(eval_recs)
refs  = [r["caption"] for r in eval_recs]

print("\nBioMedVQA ZERO-SHOT CAPTIONING -- ROCOv2 test")
metrics = compute_caption_metrics(preds, refs)

print("\nexamples:")
for r, p in list(zip(eval_recs, preds))[:5]:
    print(f"  REF : {r['caption'][:100]}")
    print(f"  PRED: {p[:100]}\n")